# Error Analysis

Load a trained checkpoint, run inference over the validation split, and inspect misclassified samples.

## Load predictions

In [ ]:
import sys
sys.path.append('..')

from torch.utils.data import DataLoader

from src.data.dataset import BreakHisDataset
from src.data.splits import filter_samples_by_patients, stratified_patient_split
from src.data.transforms import eval_transform
from src.models.classifier import BreakHisClassifier
from src.training.checkpoint import load_checkpoint
from src.training.error_analysis import collect_predictions

MAGNIFICATION = '40'
DATA_ROOT = '../data/BreaKHis_v1'
CHECKPOINT_PATH = f'../checkpoints/best_mag{MAGNIFICATION}.pt'

full_ds = BreakHisDataset(DATA_ROOT, magnification=MAGNIFICATION)
_, _, test_patients = stratified_patient_split(full_ds.samples)
val_samples = filter_samples_by_patients(full_ds.samples, test_patients)

val_ds = BreakHisDataset(DATA_ROOT, magnification=MAGNIFICATION, transform=eval_transform())
val_ds.samples = val_samples

model = BreakHisClassifier(pretrained=False)
load_checkpoint(CHECKPOINT_PATH, model)

dataloader = DataLoader(val_ds, batch_size=16)
records = collect_predictions(model, dataloader)
len(records)

## Filter errors

In [ ]:
from src.training.error_analysis import filter_misclassified, most_confident_errors

errors = filter_misclassified(records)
print(f'{len(errors)} / {len(records)} misclassified ({len(errors) / len(records):.1%})')

top_errors = most_confident_errors(records, n=9)